# VECTORBT — Backtesting ultrarrápido

Testea **miles de combinaciones** de parámetros en segundos.

## Cómo usarlo:
1. Define los rangos de parámetros que quieres probar
2. Ejecuta la celda — vectorbt los prueba todos en paralelo
3. Lee el heatmap para ver qué combinaciones son más robustas

**Regla anti-overfitting:** solo acepta combinaciones con mínimo 30 operaciones

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import vectorbt as vbt

# ── CONFIGURACION ──────────────────────────────────────────
SIMBOLO  = 'SPY'
INICIO   = '2015-01-01'
FIN      = '2026-05-01'
MIN_OPS  = 30        # minimo de operaciones para considerar valida una combinacion

# Rangos de parametros a probar (modifica estos segun tu estrategia)
EMA_RAPIDA  = range(5, 20, 2)      # 5,7,9,11,13,15,17,19
EMA_LENTA   = range(18, 40, 3)     # 18,21,24,27,30,33,36,39

print('Descargando datos...')
data  = yf.download(SIMBOLO, start=INICIO, end=FIN, progress=False, auto_adjust=True)
close = data['Close'].squeeze()
close.index = pd.to_datetime(close.index).tz_localize(None)
print(f'OK: {len(close)} dias')

In [ ]:
# ── BACKTEST MASIVO: todas las combinaciones EMA rapida x lenta ──
print(f'Probando {len(list(EMA_RAPIDA)) * len(list(EMA_LENTA))} combinaciones...')

fast_ma = vbt.MA.run(close, window=list(EMA_RAPIDA), ewm=True, short_name='fast')
slow_ma = vbt.MA.run(close, window=list(EMA_LENTA),  ewm=True, short_name='slow')

# Señales: cruce alcista
entries = fast_ma.ma_crossed_above(slow_ma)
exits   = fast_ma.ma_crossed_below(slow_ma)

# Portfolio vectorizado
pf = vbt.Portfolio.from_signals(
    close, entries, exits,
    init_cash=10000,
    fees=0.001,
    freq='1D'
)

# Filtrar combinaciones con suficientes operaciones
stats = pf.stats()
trades_counts = pf.trades.count()
mascara = trades_counts >= MIN_OPS

factor  = pf.trades.win_rate()[mascara]
retorno = pf.total_return()[mascara] * 100

print(f'Combinaciones validas (>={MIN_OPS} ops): {mascara.sum()}')

In [ ]:
# ── TOP 10 COMBINACIONES ───────────────────────────────────
ranking = retorno.sort_values(ascending=False).head(10)
print('\nTOP 10 combinaciones por retorno total:')
print('EMA_rapida | EMA_lenta | Retorno% | Trades')
print('-' * 45)
for idx, ret in ranking.items():
    n_trades = int(pf.trades.count()[idx])
    wr = float(pf.trades.win_rate()[idx]) * 100
    print(f'  {idx[0]:>8}   {idx[1]:>8}   {ret:>8.1f}%  {n_trades:>6} ops  WR:{wr:.0f}%')

In [ ]:
# ── HEATMAP: retorno por combinacion ──────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

pivot = retorno.unstack()
pivot.index.name   = 'EMA Rapida'
pivot.columns.name = 'EMA Lenta'

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns)
ax.set_yticklabels(pivot.index)
ax.set_xlabel('EMA Lenta')
ax.set_ylabel('EMA Rapida')
ax.set_title('Retorno % por combinacion EMA (verde=mejor)')
plt.colorbar(im, ax=ax, label='Retorno %')
plt.tight_layout()
out = r'C:\Users\alber\tradingview-scripts\analisis\heatmap_ema.png'
plt.savefig(out, dpi=120)
print(f'Heatmap guardado en: {out}')
plt.show()